# 01 模仿學習（Board → Action）

用啟發式教師產生的資料集訓練 IL 模型。

> 資料產生是 CPU 密集工作，**建議在本機產生**再上傳到 Drive；Colab 端只做訓練。

In [ ]:
# Colab 重啟後 cwd 會回到 /content；這裡確保在專案目錄（缺少時自動從 GitHub clone）
import os
import subprocess

try:  # Colab 重啟後 Drive 會卸載，這裡重新掛載（非 Colab 環境會跳過）
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception as exc:
    print('（非 Colab 環境或已掛載，略過）', exc)

PROJECT = '/content/tetrio-ai'
REPO_URL = 'https://github.com/wallacechen0130/tetr_bot.git'
if not os.path.exists(os.path.join(PROJECT, 'requirements.txt')):
    subprocess.run(f'git clone --depth 1 {REPO_URL} {PROJECT}', shell=True, check=True)
os.chdir(PROJECT)
print('工作目錄:', os.getcwd())

In [ ]:
import json
import os

import glob

os.environ.setdefault('TETRIO_AI_DRIVE', '/content/drive/MyDrive/tetrio-ai')
DRIVE_ROOT = os.environ['TETRIO_AI_DRIVE']
DATASETS_DIR = os.path.join(DRIVE_ROOT, 'datasets')

# 優先使用 heuristic-v1；沒有就用 Drive 上最新的資料集（例如 verify-v2）
preferred = os.path.join(DATASETS_DIR, 'heuristic-v1')
found = sorted(p for p in glob.glob(os.path.join(DATASETS_DIR, '*')) if os.path.isdir(p))
DATA_ROOT = preferred if os.path.isdir(preferred) else (found[-1] if found else preferred)
if found and DATA_ROOT != preferred:
    print('找不到 heuristic-v1，改用 Drive 上最新的資料集:', os.path.basename(DATA_ROOT))
CKPT_ROOT = os.path.join(DRIVE_ROOT, 'checkpoints', 'il')
os.makedirs(CKPT_ROOT, exist_ok=True)
print('dataset:', DATA_ROOT)
print('checkpoints:', CKPT_ROOT)
MANIFEST = os.path.join(DATA_ROOT, 'manifest.json')
print(json.load(open(MANIFEST, encoding='utf-8')) if os.path.exists(MANIFEST) else '尚未有資料集（可用下一個 cell 產生）')

In [ ]:
# （可選）若 Drive 上還沒有資料集，直接在 Colab 產生一份小的
if not os.path.exists(DATA_ROOT):
    !python -m scripts.generate_dataset --out {DATA_ROOT} --episodes 40 --workers 4 --max-pieces 200

In [ ]:
# 先用小樣本確認 loss 會下降，再放大到全量
!python -m scripts.train_il --data-root {DATA_ROOT} --epochs 3 --limit 20000 \
    --checkpoint-dir {CKPT_ROOT} --network resnet --out {CKPT_ROOT}/il_small.json

In [ ]:
# 正式訓練（全量資料，約 30 epochs + early stopping）
!python -m scripts.train_il --data-root {DATA_ROOT} --epochs 30 \
    --checkpoint-dir {CKPT_ROOT} --network resnet --out {CKPT_ROOT}/il_full.json

In [ ]:
# 驗收：top-1 ≥ 0.85、top-3 ≥ 0.97（configs/il.yaml 的 metrics 門檻）
print(json.load(open(os.path.join(CKPT_ROOT, 'il_full.json'), encoding='utf-8'))['best_top1'])